<a href="https://colab.research.google.com/github/shivyadev/Learning_RAG/blob/master/Learning_RAG.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Chunking Techniques

#### Overlapping Chunks

In [1]:
%pip install -q sentence-transformers

## Loading the required models

In [2]:
from sentence_transformers import SentenceTransformer, CrossEncoder, util
import nltk
from nltk.tokenize import sent_tokenize
import re

nltk.download("punkt_tab")

model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2") # For making embeddings of sentences
reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2") # Model which will be used for reranking

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/1.33k [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

In [3]:
documents = [
    {
        "title": "Gradient Descent",
        "text": """
        A machine learning model has parameters that change during training.
        A loss function measures how wrong the model's predictions are.
        Gradient descent uses gradients to determine how changes to model
        parameters affect the loss.
        It updates parameters in the direction that reduces the loss.
        The learning rate controls the size of each parameter update.
        A learning rate that is too large can cause the model to overshoot
        a good solution, while a learning rate that is too small can make
        training very slow.
        """
    },
    {
        "title": "Neural Networks",
        "text": """
        Neural networks consist of interconnected layers of artificial neurons.
        Each neuron receives numerical inputs, applies weights and a bias,
        and uses an activation function to produce an output.
        During training, neural networks learn patterns from examples by
        adjusting their weights and biases.
        The input layer receives the data, hidden layers process it, and
        the output layer produces the prediction.
        Deep learning uses neural networks with multiple hidden layers.
        These layers can learn increasingly complex patterns from data.
        """
    },
    {
        "title": "Transformers",
        "text": """
        Transformer models use attention mechanisms to represent relationships
        between tokens in a sequence.
        Attention allows a model to assign different importance to different
        tokens when processing a particular token.
        Transformers are widely used in natural language processing tasks,
        including text generation, translation, summarization, and semantic search.
        """
    }
]

##### Function for creating semantic chunks

In [4]:
def create_semantic_chunks(documents, similarity_threshold=0.45):
    semantic_chunks = []

    for doc in documents:
        sentences = sent_tokenize(doc["text"].strip())
        sentence_embeddings = model.encode(sentences)

        current_chunk = [sentences[0]]

        for i in range(1, len(sentences)):
            similarity = util.cos_sim(
                sentence_embeddings[i - 1],
                sentence_embeddings[i]
            ).item()

            if similarity >= similarity_threshold:
                current_chunk.append(sentences[i])
            else:
                semantic_chunks.append({
                    "title": doc["title"],
                    "text": " ".join(current_chunk)
                })
                current_chunk = [sentences[i]]

        if current_chunk:
            semantic_chunks.append({
                "title": doc["title"],
                "text": " ".join(current_chunk)
            })

    return semantic_chunks

In [5]:
semantic_chunks = create_semantic_chunks(documents)

print("Total semantic chunks:", len(semantic_chunks))

Total semantic chunks: 6


In [6]:
semantic_chunk_texts = [
    chunk["text"] for chunk in semantic_chunks
]

semantic_chunk_embeddings = model.encode(semantic_chunk_texts)

print("Chunks:", len(semantic_chunks))
print("Embeddings:", len(semantic_chunk_embeddings))

Chunks: 6
Embeddings: 6


##### Function to score the chunks based on the relevance

In [7]:
def search_document(query, top_k=4):
    query_embedding = model.encode(query)

    similarities = util.cos_sim(
        query_embedding,
        semantic_chunk_embeddings
    )[0]

    ranked_indices = similarities.argsort(descending=True)

    results = []

    for idx in ranked_indices[:top_k]:
        i = idx.item()

        results.append({
            "title": semantic_chunks[i]["title"],
            "text": semantic_chunks[i]["text"],
            "score": similarities[i].item()
        })

    return results

In [8]:
query = "What do hidden layers do in deep learning?"

candidates = search_document(query, top_k=4)

for i, result in enumerate(candidates, start=1):
    print(f"\nCandidate {i} — {result['title']}")
    print(f"Initial score: {result['score']:.4f}")
    print(result["text"])


Candidate 1 — Neural Networks
Initial score: 0.7441
The input layer receives the data, hidden layers process it, and
        the output layer produces the prediction. Deep learning uses neural networks with multiple hidden layers. These layers can learn increasingly complex patterns from data.

Candidate 2 — Neural Networks
Initial score: 0.5540
Neural networks consist of interconnected layers of artificial neurons. Each neuron receives numerical inputs, applies weights and a bias,
        and uses an activation function to produce an output. During training, neural networks learn patterns from examples by
        adjusting their weights and biases.

Candidate 3 — Gradient Descent
Initial score: 0.4136
A loss function measures how wrong the model's predictions are. Gradient descent uses gradients to determine how changes to model
        parameters affect the loss. It updates parameters in the direction that reduces the loss. The learning rate controls the size of each parameter updat

##### Forming pairs to feed into reranker

In [9]:
pairs = [(query, result['text']) for result in candidates]

##### Reranking the pairs using reranker cross-encoder model

In [10]:
reranked_scores = reranker.predict(pairs)

##### Reranking results based on the new scores

In [11]:
reranked_results = []

for i, result in enumerate(candidates):
    reranked_results.append({
        **result,
        "rerank_score": float(reranked_scores[i])
    })

reranked_results.sort(
    key=lambda result: result["rerank_score"],
    reverse=True
)

for i, result in enumerate(reranked_results, start=1):
    print(f"\nRank {i} — {result['title']}")
    print(f"Initial score: {result['score']:.4f}")
    print(f"Reranker score: {result['rerank_score']:.4f}")
    print(result["text"])


Rank 1 — Neural Networks
Initial score: 0.7441
Reranker score: 8.3028
The input layer receives the data, hidden layers process it, and
        the output layer produces the prediction. Deep learning uses neural networks with multiple hidden layers. These layers can learn increasingly complex patterns from data.

Rank 2 — Neural Networks
Initial score: 0.5540
Reranker score: -7.1261
Neural networks consist of interconnected layers of artificial neurons. Each neuron receives numerical inputs, applies weights and a bias,
        and uses an activation function to produce an output. During training, neural networks learn patterns from examples by
        adjusting their weights and biases.

Rank 3 — Gradient Descent
Initial score: 0.4136
Reranker score: -10.6897
A loss function measures how wrong the model's predictions are. Gradient descent uses gradients to determine how changes to model
        parameters affect the loss. It updates parameters in the direction that reduces the loss. Th

##### Chunk rankings

In [12]:
for i, chunk in enumerate(semantic_chunks):
    print(f"\nChunk {i}")
    print("Title:", chunk["title"])
    print(chunk["text"])


Chunk 0
Title: Gradient Descent
A machine learning model has parameters that change during training.

Chunk 1
Title: Gradient Descent
A loss function measures how wrong the model's predictions are. Gradient descent uses gradients to determine how changes to model
        parameters affect the loss. It updates parameters in the direction that reduces the loss. The learning rate controls the size of each parameter update. A learning rate that is too large can cause the model to overshoot
        a good solution, while a learning rate that is too small can make
        training very slow.

Chunk 2
Title: Neural Networks
Neural networks consist of interconnected layers of artificial neurons. Each neuron receives numerical inputs, applies weights and a bias,
        and uses an activation function to produce an output. During training, neural networks learn patterns from examples by
        adjusting their weights and biases.

Chunk 3
Title: Neural Networks
The input layer receives the dat

In [13]:
relevant_chunk_ids = {3}

##### Evaluating the retrieval

In [14]:
for rank, result in enumerate(candidates, start=1):
    chunk_id = next(
        i for i, chunk in enumerate(semantic_chunks)
        if chunk["text"] == result["text"]
    )

    relevant = chunk_id in relevant_chunk_ids

    print(
        f"Rank: {rank} | "
        f"Chunk: {chunk_id} | "
        f"Score: {result['score']:.4f} | "
        f"Relevant: {relevant}"
    )

Rank: 1 | Chunk: 3 | Score: 0.7441 | Relevant: True
Rank: 2 | Chunk: 2 | Score: 0.5540 | Relevant: False
Rank: 3 | Chunk: 1 | Score: 0.4136 | Relevant: False
Rank: 4 | Chunk: 0 | Score: 0.3221 | Relevant: False


In [15]:
relevance = [True, False, False, False]

k = 4

precision_at_k = sum(relevance[:k]) / k

total_relevant = len(relevant_chunk_ids)
recall_at_k = sum(relevance[:k]) / total_relevant

reciprocal_rank = 0
for rank, is_relevant in enumerate(relevance, start=1):
    if is_relevant:
        reciprocal_rank = 1 / rank
        break

print("Precision@4:", precision_at_k)
print("Recall@4:", recall_at_k)
print("Reciprocal Rank:", reciprocal_rank)

Precision@4: 0.25
Recall@4: 1.0
Reciprocal Rank: 1.0


##### Evaluating reranker result

In [16]:
reranked_results = []

for result, score in zip(candidates, reranked_scores):
    chunk_id = next(
        i for i, chunk in enumerate(semantic_chunks)
        if chunk["text"] == result["text"]
    )

    reranked_results.append({
        "chunk_id": chunk_id,
        "text": result["text"],
        "rerank_score": float(score),
        "relevant": chunk_id in relevant_chunk_ids
    })

reranked_results.sort(
    key=lambda result: result["rerank_score"],
    reverse=True
)

for rank, result in enumerate(reranked_results, start=1):
    print(
        f"Rank: {rank} | "
        f"Chunk: {result['chunk_id']} | "
        f"Score: {result['rerank_score']:.4f} | "
        f"Relevant: {result['relevant']}"
    )

Rank: 1 | Chunk: 3 | Score: 8.3028 | Relevant: True
Rank: 2 | Chunk: 2 | Score: -7.1261 | Relevant: False
Rank: 3 | Chunk: 1 | Score: -10.6897 | Relevant: False
Rank: 4 | Chunk: 0 | Score: -10.8428 | Relevant: False


##### Testing the model across different queries

In [17]:
test_cases = [
    {
        "query": "What is gradient descent?",
        "relevant_ids": {0, 1}
    },
    {
        "query": "Why can a learning rate that is too large be a problem?",
        "relevant_ids": {1}
    },
    {
        "query": "What do hidden layers do in deep learning?",
        "relevant_ids": {3}
    },
    {
        "query": "How do neural networks learn?",
        "relevant_ids": {2}
    },
    {
        "query": "What does attention do in transformers?",
        "relevant_ids": {4}
    }
]

In [18]:
def evaluate_ranking(results, relevant_ids, k=4):
    results = results[:k]

    relevance = [
        result["chunk_id"] in relevant_ids
        for result in results
    ]

    precision = sum(relevance) / k
    recall = sum(relevance) / len(relevant_ids)

    reciprocal_rank = 0
    for rank, is_relevant in enumerate(relevance, start=1):
        if is_relevant:
            reciprocal_rank = 1 / rank
            break

    return precision, recall, reciprocal_rank

In [19]:
evaluation_results = []

for case in test_cases:
    query = case["query"]
    relevant_ids = case["relevant_ids"]

    # Retrieve the initial candidates
    candidates = search_document(query, top_k=4)

    # Attach each candidate's chunk ID
    for result in candidates:
        result["chunk_id"] = next(
            i for i, chunk in enumerate(semantic_chunks)
            if chunk["text"] == result["text"]
        )

    # Evaluate the original semantic ranking
    semantic_metrics = evaluate_ranking(
        candidates, relevant_ids
    )

    # Rerank the same candidates
    pairs = [
        (query, result["text"])
        for result in candidates
    ]
    scores = reranker.predict(pairs)

    reranked = [
        {**result, "rerank_score": float(score)}
        for result, score in zip(candidates, scores)
    ]

    reranked.sort(
        key=lambda result: result["rerank_score"],
        reverse=True
    )

    # Evaluate the reranked ranking
    reranked_metrics = evaluate_ranking(
        reranked, relevant_ids
    )

    evaluation_results.append({
        "query": query,
        "semantic": semantic_metrics,
        "reranked": reranked_metrics
    })

    print(f"\nQuery: {query}")
    print("Semantic metrics (P@4, R@4, RR):", semantic_metrics)
    print("Reranked metrics (P@4, R@4, RR):", reranked_metrics)



Query: What is gradient descent?
Semantic metrics (P@4, R@4, RR): (0.5, 1.0, 1.0)
Reranked metrics (P@4, R@4, RR): (0.5, 1.0, 1.0)

Query: Why can a learning rate that is too large be a problem?
Semantic metrics (P@4, R@4, RR): (0.25, 1.0, 1.0)
Reranked metrics (P@4, R@4, RR): (0.25, 1.0, 1.0)

Query: What do hidden layers do in deep learning?
Semantic metrics (P@4, R@4, RR): (0.25, 1.0, 1.0)
Reranked metrics (P@4, R@4, RR): (0.25, 1.0, 1.0)

Query: How do neural networks learn?
Semantic metrics (P@4, R@4, RR): (0.25, 1.0, 1.0)
Reranked metrics (P@4, R@4, RR): (0.25, 1.0, 1.0)

Query: What does attention do in transformers?
Semantic metrics (P@4, R@4, RR): (0.25, 1.0, 1.0)
Reranked metrics (P@4, R@4, RR): (0.25, 1.0, 1.0)


In [20]:
# Average the metrics across all test questions
semantic_average = tuple(
    sum(result["semantic"][i] for result in evaluation_results)
    / len(evaluation_results)
    for i in range(3)
)

reranked_average = tuple(
    sum(result["reranked"][i] for result in evaluation_results)
    / len(evaluation_results)
    for i in range(3)
)

print("\n--- AVERAGE METRICS ---")
print("Semantic (P@4, R@4, MRR):", semantic_average)
print("Reranked (P@4, R@4, MRR):", reranked_average)


--- AVERAGE METRICS ---
Semantic (P@4, R@4, MRR): (0.3, 1.0, 1.0)
Reranked (P@4, R@4, MRR): (0.3, 1.0, 1.0)


#### RAG complete pipeline

In [21]:
def retrieve_context(query, top_k=3):
    results = search_document(query, top_k=top_k)

    context = "\n\n".join(
        f"Source: {result['title']}\n{result['text']}"
        for result in results
    )

    return context, results

In [22]:
query = "What does gradient descent do?"

context, results = retrieve_context(query)

print(context)

Source: Gradient Descent
A loss function measures how wrong the model's predictions are. Gradient descent uses gradients to determine how changes to model
        parameters affect the loss. It updates parameters in the direction that reduces the loss. The learning rate controls the size of each parameter update. A learning rate that is too large can cause the model to overshoot
        a good solution, while a learning rate that is too small can make
        training very slow.

Source: Neural Networks
The input layer receives the data, hidden layers process it, and
        the output layer produces the prediction. Deep learning uses neural networks with multiple hidden layers. These layers can learn increasingly complex patterns from data.

Source: Gradient Descent
A machine learning model has parameters that change during training.


In [23]:
%pip install -q transformers

In [24]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

model_name = "google/flan-t5-base"

tokenizer = AutoTokenizer.from_pretrained(model_name)
generator_model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

config.json:   0%|          | 0.00/1.40k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.54k [00:00<?, ?B/s]

spiece.model: reconstructing file:   0%|          |  0.00B /  792kB            

spiece.model: downloading bytes:           |  0.00B            

tokenizer.json:   0%|          | 0.00/2.42M [00:00<?, ?B/s]

...

model.safetensors: reconstructing file:   0%|          |  0.00B /  990MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

In [25]:
def generate_answer(query, context):

    prompt=f"""
    Answer the question using the context below.
If the context does not contain the answer, say you do not have enough information.

Context:
{context}

Question: {query}
Answer:
"""
    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length = 512
    )

    outputs = generator_model.generate(
        **inputs,
        max_new_tokens=100,
        do_sample=False,
    )

    answer = tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )

    return answer

In [26]:
query = "What does gradient descent do?"

context, results = retrieve_context(query)
answer = generate_answer(query, context)

print("Question:", query)
print("\nRetrieved context:\n", context)
print("\nGenerated answer:\n", answer)

Question: What does gradient descent do?

Retrieved context:
 Source: Gradient Descent
A loss function measures how wrong the model's predictions are. Gradient descent uses gradients to determine how changes to model
        parameters affect the loss. It updates parameters in the direction that reduces the loss. The learning rate controls the size of each parameter update. A learning rate that is too large can cause the model to overshoot
        a good solution, while a learning rate that is too small can make
        training very slow.

Source: Neural Networks
The input layer receives the data, hidden layers process it, and
        the output layer produces the prediction. Deep learning uses neural networks with multiple hidden layers. These layers can learn increasingly complex patterns from data.

Source: Gradient Descent
A machine learning model has parameters that change during training.

Generated answer:
 It updates parameters in the direction that reduces the loss


## Main Pipeline Function

In [33]:
def rag_pipeline(query, top_k=3, threshold=0.4):
    context, results = retrieve_context(query, top_k=top_k)

    # Check the highest similarity score
    best_score = results[0]["score"]

    if best_score < threshold: # Threshold to reject answer if score is too low
        return {
            "answer": "I don't have enough relevant information to answer that from my documents.",
            "sources": results,
            "best_score": best_score
        }

    answer = generate_answer(query, context)

    return {
        "answer": answer,
        "sources": results,
        "best_score": best_score
    }

In [28]:
result = rag_pipeline("Who invented computers?")

print("Answer:", result["answer"])
print("\nSources:")

for source in result["sources"]:
    print(f"- {source['title']}: {source['text']}")

Answer: not enough information

Sources:
- Neural Networks: Neural networks consist of interconnected layers of artificial neurons. Each neuron receives numerical inputs, applies weights and a bias,
        and uses an activation function to produce an output. During training, neural networks learn patterns from examples by
        adjusting their weights and biases.
- Neural Networks: The input layer receives the data, hidden layers process it, and
        the output layer produces the prediction. Deep learning uses neural networks with multiple hidden layers. These layers can learn increasingly complex patterns from data.
- Gradient Descent: A machine learning model has parameters that change during training.


In [31]:
queries = [
    "What does gradient descent do?",
    "Who invented the first computer?"
]

for query in queries:
    results = search_document(query, top_k=3)

    print("\nQuestion:", query)

    for result in results:
        print(
            f"Score: {result['score']:.3f} | "
            f"Source: {result['title']}"
        )


Question: What does gradient descent do?
Score: 0.697 | Source: Gradient Descent
Score: 0.422 | Source: Neural Networks
Score: 0.388 | Source: Gradient Descent

Question: Who invented the first computer?
Score: 0.095 | Source: Neural Networks
Score: 0.089 | Source: Neural Networks
Score: 0.075 | Source: Gradient Descent


In [32]:
queries = [
    "What does gradient descent do?",
    "Who invented the first computer?"
]

for query in queries:
    result = rag_pipeline(query)

    print("\nQuestion:", query)
    print("Best score:", round(result["best_score"], 3))
    print("Answer:", result["answer"])


Question: What does gradient descent do?
Best score: 0.697
Answer: It updates parameters in the direction that reduces the loss

Question: Who invented the first computer?
Best score: 0.095
Answer: I don't have enough relevant information to answer that from my documents.
